# 🤖 Notebook 02 — MLP com PyTorch

## 🎯 Objetivo
Implementar uma rede neural (MLP) para previsão de churn e comparar com os modelos baseline.

---

## 📦 1. Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, precision_score, recall_score, f1_score

import mlflow

## 📊 2. Carregamento dos dados

In [2]:
DATA_PATH = Path("../data/raw/Telco_customer_churn.xlsx")

if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/Telco_customer_churn.xlsx")

df = pd.read_excel(DATA_PATH, engine="openpyxl")

df.columns = df.columns.str.strip()

cols_to_drop = [
    "CustomerID",
    "Count",
    "Churn Label",
    "Churn Score",
    "Churn Reason",
    "Lat Long",
]

df = df.drop(columns=cols_to_drop, errors="ignore")

target = "Churn Value"

## 🧹 3. Preparação dos dados

In [3]:
# Conversão importante
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")

# Conversão Yes/No
for col in df.select_dtypes(include="object").columns:
    if set(df[col].dropna().unique()) == {"Yes", "No"}:
        df[col] = df[col].map({"Yes": 1, "No": 0})

X = df.drop(columns=[target]).copy()
y = df[target].copy()

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

X[cat_cols] = X[cat_cols].astype(str)

## 🔀 4. Train/Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## ⚙️ 5. Pipeline de pré-processamento

In [5]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols),
])

## 🔄 6. Transformação para PyTorch

In [6]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Converter sparse → dense
X_train_tensor = torch.tensor(X_train_processed.toarray(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_processed.toarray(), dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

## 🤖 7. Definição da MLP

In [7]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

## ⚙️ 8. Inicialização

In [8]:
input_dim = X_train_tensor.shape[1]

model = MLP(input_dim)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## 🔁 9. Treinamento

In [9]:
epochs = 50

for epoch in range(epochs):
    model.train()
    
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch} - Loss: {loss.item():.4f}")

Epoch 0 - Loss: 0.6600
Epoch 10 - Loss: 0.6249
Epoch 20 - Loss: 0.5600
Epoch 30 - Loss: 0.4814
Epoch 40 - Loss: 0.4321


## 📊 10. Avaliação

In [10]:
model.eval()

with torch.no_grad():
    y_pred_prob = model(X_test_tensor).numpy()

threshold = 0.3
y_pred = (y_pred_prob >= threshold).astype(int)

print(classification_report(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_pred_prob))

              precision    recall  f1-score   support

           0       0.89      0.77      0.83      1035
           1       0.54      0.74      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.76      0.77      1409

AUC: 0.8340618460823064


## 📈 11. Métricas

In [11]:
mlp_metrics = {
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "auc_roc": roc_auc_score(y_test, y_pred_prob),
}

mlp_metrics

{'precision': 0.5398058252427185,
 'recall': 0.7433155080213903,
 'f1': 0.625421822272216,
 'auc_roc': 0.8340618460823064}

## 🧪 12. MLflow

In [12]:
mlflow.set_experiment("telco_churn_baselines")

with mlflow.start_run(run_name="mlp_v1"):
    mlflow.log_param("model", "MLP")
    mlflow.log_param("epochs", 50)
    mlflow.log_param("lr", 0.001)
    
    for k, v in mlp_metrics.items():
        mlflow.log_metric(k, float(v))

## 🏆 13. Comparação com Baseline

| Modelo                    | Accuracy | Precision | Recall | F1-score | AUC-ROC |
|--------------------------|---------|----------|--------|----------|---------|
| Logistic Regression (0.3)| 0.79    | 0.54     | 0.74   | 0.623    | **0.84** |
| MLP (PyTorch)            | 0.76    | 0.54     | 0.74   | **0.625**| 0.83    |
| Random Forest            | 0.80    | **0.68** | 0.48   | 0.56     | 0.84    |
| Dummy Classifier         | 0.73    | 0.00     | 0.00   | 0.00     | -       |

## 📌 14. Conclusão

A rede neural (MLP) apresentou desempenho muito próximo ao da Regressão Logística, com leve ganho em F1-score, porém sem melhoria significativa na capacidade de discriminação (AUC-ROC).

Esse resultado sugere que, para este dataset, modelos lineares já são suficientes para capturar os padrões relevantes, possivelmente devido ao tamanho limitado dos dados e à natureza das variáveis.

Dessa forma, optou-se por manter a Regressão Logística como modelo principal, considerando sua simplicidade, interpretabilidade e desempenho competitivo.